# TSIS &mdash; Lab 3

## Introduction to LangChain, LangGraph, and Deep Agents

Welcome to the third lab on _Special Topics in Software Engineering_! Now that we have grown familiar with large language models and their capabilities, it's time to see how we can use them in more complex workflows.

As we've previously seen, the API provided by LLM providers is pretty simple. We send a **list of messages** (a _conversation_), provide the model with a list of **tools to call** and possibly request the **output to follow a certain structure**, and that's about it. Ignoring more advanced features such as fine-tuning, computing text embeddings, uploading files or processing audio, we can still achieve quite a lot with just the basic text-completion interface &mdash; as long as we use it wisely!

**Prerequisites:** make sure you have completed [Lab 1](lab1_bis.ipynb) (conversations, structured outputs, tool calling) and [Lab 2](lab2_rag.ipynb) (RAG over the Atlas project handbook).

## Learning objectives

By the end of this lab you should be able to:

- interact with language models through LangChain's **provider-agnostic interface** ([`ChatOpenAI`](https://reference.langchain.com/python/langchain-openai), [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model), and standard message types);
- augment a chat model with Python functions using the [`@tool`](https://reference.langchain.com/python/langchain-core/tools/tool) decorator (`bind_tools`) and enforce schemas with [`with_structured_output`](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/with_structured_output);
- build a tool-calling [ReAct](https://arxiv.org/abs/2210.03629) agent with [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) and inspect its full message trajectory;
- design explicit **state machines** in [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) using `StateGraph`, conditional routing, in-memory checkpointing (`InMemorySaver`), and **human-in-the-loop** interrupts (`interrupt` / `Command`);
- configure an autonomous **Deep Agent** ([`create_deep_agent`](https://docs.langchain.com/oss/python/deepagents/overview)) with sandboxed filesystem backends, least-privilege permissions, and specialized subagents to research the local Atlas project documentation.

## Setup

Run the following cell to ensure you have all of the required packages installed on your machine.

In [ ]:
%pip install --quiet --upgrade deepagents langgraph langchain langchain-openai pydantic

Remember to **fill in** below the API key your team requested (or ask the teaching assistant for one to use during the lab).

In [ ]:
API_KEY = "FILL IN YOUR API KEY HERE"
BASE_URL = "https://api.llm.acc.unibuc.ro/v1"
MODEL_NAME = "Qwen3.8-27B"

## Provider-agnostic model API

In Lab 1 we saw two different vendor SDKs (`google-genai` in `lab1.ipynb` and `openai` in `lab1_bis.ipynb`). While both expose similar concepts, their method names, message formats, and tool schemas differ.

A big advantage of using LangChain comes from its **provider-independent standard interfaces**. LangChain has [an ecosystem of 1000+ integrations](https://docs.langchain.com/oss/python/integrations/providers/overview), which provide access to chat models, embedding models, tools, checkpointers / stores etc.

For example, we'll use the [`langchain-openai`](https://reference.langchain.com/python/langchain-openai) package to call our OpenAI API-compatible endpoint. This could easily be switched with the [`langchain-anthropic`](https://reference.langchain.com/python/langchain-anthropic) package if we wanted to use Claude, or `langchain-google-genai` for Gemini. Note that the [base interface](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel) is defined in the `langchain-core` package; LangChain defines an interface contract which implementations must abide by.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=MODEL_NAME,
    # Reduce the generation temperature to make the model more deterministic.
    temperature=0,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# Make an API call to check it works
print("Available models:")
for item in model.root_client.models.list():
    print(item.id)

No matter which LLM provider we use, you can invoke it by using the [`invoke()`](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/invoke) method:

In [ ]:
response = model.invoke(
    "Why is Romania a great country?",
    # This should be a no-brainer!
    reasoning_effort="none",
    # Keep it brief.
    max_tokens=200,
)
print(response.content)

A more succinct method of initializing a chat model, without explicitly importing a provider's integration library, is by using the [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) function:

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=f"openai:{MODEL_NAME}",
    # Same parameters as before
    temperature=0,
    api_key=API_KEY,
    base_url=BASE_URL,
)

response = model.invoke("Why is the sky blue?")
print(response.content[:200])

### Standard message types

When passing a string to `model.invoke("...")`, LangChain automatically wraps it in a [`HumanMessage`](https://reference.langchain.com/python/langchain-core/messages/human/HumanMessage). For multi-turn conversations or system instructions, we pass a list of message objects (or OpenAI-style `{"role": ..., "content": ...}` dictionaries):

- `SystemMessage`: instructions that set the assistant's persona, constraints, and behavior;
- `HumanMessage`: input from the end user;
- `AIMessage`: response from the model (including `content`, `tool_calls`, and `usage_metadata`);
- `ToolMessage`: the output returned by our application after executing a tool call.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a concise software architecture reviewer. Reply in at most 2 bullet points."),
    HumanMessage(content="Should our startup begin with a monolith or 15 microservices?"),
]

ai_msg = model.invoke(messages)
print(type(ai_msg))
print(ai_msg.content)
print("\nToken usage:", ai_msg.usage_metadata)

## Augmenting the LLM

Before building full agents or multi-step graphs, we typically **augment** a base LLM with two capabilities we met in Lab 1:

1. **Tool calling** &mdash; letting the model request deterministic function calls;
2. **Structured outputs** &mdash; constraining the model's response to a validated schema.

LangChain eliminates the manual JSON Schema plumbing from Lab 1 by inferring schemas directly from Python type annotations, docstrings, and Pydantic models.

### Giving it tools

In Lab 1, we had to write a verbose JSON schema dictionary (`{"type": "function", "name": ..., "parameters": ...}`) by hand for every tool. In LangChain, we decorate a typed Python function with [`@tool`](https://reference.langchain.com/python/langchain-core/tools/tool):

- The **function name** becomes the tool's name;
- The **docstring** becomes the tool's description (crucial so the LLM knows *when* and *why* to call it!);
- The **type annotations** become the JSON Schema for the tool's arguments.

Let's start with a simple calculator tool that computes how many minutes of downtime a service-level agreement (SLA) allows per 30-day month.

In [ ]:
from langchain.tools import tool


@tool
def calculate_monthly_downtime_budget(sla_percentage: float) -> str:
    """Calculate the maximum allowed downtime in minutes per 30-day month for a given SLA percentage (e.g. 99.9)."""
    if not (0.0 < sla_percentage <= 100.0):
        raise ValueError("sla_percentage must be between 0 and 100")
    total_minutes = 30 * 24 * 60
    allowed_minutes = total_minutes * (1.0 - sla_percentage / 100.0)
    return f"{allowed_minutes:.2f} minutes per 30-day month"


print("Tool name:       ", calculate_monthly_downtime_budget.name)
print("Tool description:", calculate_monthly_downtime_budget.description)
print("Tool args schema:", calculate_monthly_downtime_budget.args)

To make the model aware of one or more tools, we call [`model.bind_tools([...])`](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/bind_tools). This returns a new runnable model wrapper that automatically attaches the tool declarations to every API request.

Just like in Lab 1, **calling `model_with_tools.invoke(...)` does not execute the Python function**. Instead, the returned `AIMessage` contains a `tool_calls` list describing which tool the model wants to run and with what arguments.

In [ ]:
model_with_tools = model.bind_tools([calculate_monthly_downtime_budget])

query = "Our payment service promises a 99.95% SLA. How many minutes of downtime can we afford in a 30-day month?"
ai_msg = model_with_tools.invoke(query)

print("Direct text content:", repr(ai_msg.content))
print("Requested tool calls:", ai_msg.tool_calls)

When we pass a tool call dictionary directly to `tool.invoke(tool_call)`, LangChain executes the underlying Python function and wraps the result in a `ToolMessage` (automatically carrying over the `tool_call_id` so the LLM can match the observation to its request):

In [ ]:
if ai_msg.tool_calls:
    tool_msg = calculate_monthly_downtime_budget.invoke(ai_msg.tool_calls[0])
    print("ToolMessage:", tool_msg)

    # Send the full conversation (question -> tool call -> tool result) back to the model
    final_response = model_with_tools.invoke([HumanMessage(content=query), ai_msg, tool_msg])
    print("\nFinal answer:")
    print(final_response.content)

### Structured outputs

When we want the LLM's output to adhere to a strict schema (for example, parsing a raw incident report or a pull-request description into a typed Python object), we use [`model.with_structured_output(Schema)`](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel/with_structured_output).

Compare the code below with the manual `BugTriageReport.model_json_schema()` / `model_validate_json()` plumbing from Lab 1 &mdash; LangChain handles the schema conversion and validation automatically and returns an instance of our Pydantic model:

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class BugTriageReport(BaseModel):
    """Structured bug triage data extracted from raw incident reports."""

    summary: str = Field(description="Concise 1-sentence technical summary of the issue")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"] = Field(
        description="Severity level of the issue"
    )
    affected_component: Literal["auth", "database", "frontend", "payment", "worker", "unknown"] = (
        Field(description="Component of the system affected by the issue")
    )
    reproduction_steps: list[str] = Field(description="Ordered list of steps to reproduce")
    requires_immediate_rollback: bool = Field(
        description="Whether the issue requires an immediate rollback of the deployment"
    )


structured_model = model.with_structured_output(BugTriageReport)

raw_ticket = """
Hey team, ever since the v2.4 deploy 10 minutes ago, users in EU cannot complete checkout!
When they click 'Pay Now' with Stripe, the spinner hangs forever and the console logs a 502 Bad Gateway
on /api/v2/charge. Tested on Chrome and Firefox. We are losing orders right now!
"""

report: BugTriageReport = structured_model.invoke(
    f"Extract structured bug triage data from this incident report:\n\n{raw_ticket}"
)

print(type(report))
print(f"Summary:   {report.summary}")
print(f"Severity:  {report.severity} | Rollback? {report.requires_immediate_rollback}")
print(f"Component: {report.affected_component}")
print(f"Steps:     {report.reproduction_steps}")

### Mission 1 &mdash; Define a tool and a structured schema (guided)

**Goal:** create a safe, read-only tool and a Pydantic schema for reviewing dependencies.

Complete the blanks (`...`) in the cell below:

1. Decorate `get_package_advisory` with `@tool` and give it a clear docstring so an LLM knows what it returns.
2. Add a field `recommended_action` to `DependencyReview` restricted to `Literal["upgrade", "keep", "investigate"]`.
3. Run `check_mission_1()` to verify your tool and schema definitions.

In [ ]:
ADVISORIES = {
    "pyjwt": {"installed": "1.7.1", "latest_patched": "2.9.0", "cve": "CVE-2022-29217", "severity": "HIGH"},
    "pydantic": {"installed": "2.14.1", "latest_patched": "2.14.1", "cve": None, "severity": "NONE"},
}


# TODO 1: decorate this function as a LangChain tool and write a helpful docstring.
def get_package_advisory(package_name: str) -> str:
    """..."""
    pkg = package_name.lower().strip()
    if pkg not in ADVISORIES:
        return f"No advisory entry found for package '{package_name}'."
    return str(ADVISORIES[pkg])


class DependencyReview(BaseModel):
    """Structured security assessment for a Python dependency."""

    package_name: str = Field(description="Name of the inspected package")
    is_vulnerable: bool = Field(description="True if the installed version has a known CVE")
    # TODO 2: restrict recommended_action to Literal["upgrade", "keep", "investigate"]
    recommended_action: ... = Field(description="Action the engineering team should take")


def check_mission_1():
    assert hasattr(get_package_advisory, "name"), "Decorate get_package_advisory with @tool"
    assert get_package_advisory.description and get_package_advisory.description != "...", (
        "Provide a meaningful docstring for get_package_advisory"
    )
    schema_props = DependencyReview.model_json_schema()["properties"]
    assert schema_props["recommended_action"].get("enum") == ["upgrade", "keep", "investigate"], (
        "recommended_action should be a Literal['upgrade', 'keep', 'investigate']"
    )
    print("Mission 1 check passed!")


# Complete the TODOs above, then uncomment the next line:
# check_mission_1()

## Your first "agent"

What is the difference between a single augmented LLM call and an **agent**?

In an augmented LLM call, our code makes a single request and stops. In an **agent**, the LLM runs inside a loop (often called the [**ReAct**](https://arxiv.org/abs/2210.03629) loop: **Re**ason $\rightarrow$ **Act** $\rightarrow$ **Observe**):

1. The model inspects the conversation history and decides whether it needs more information;
2. If it emits one or more `tool_calls`, the harness executes those tools and appends the resulting `ToolMessage`s to the conversation;
3. The model is invoked again with the new observations, repeating until it has enough evidence to produce a final response.

### Creating a new LangChain agent

In modern LangChain (`v1.x`), [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) is the standard factory for building a tool-calling agent (backed under the hood by the LangGraph runtime).

Let's equip an SRE incident-response agent with three deterministic, read-only tools that inspect production telemetry, recent deployments, and error logs for our services:

In [ ]:
import json

from langchain.agents import create_agent


@tool
def get_service_health(service_name: str) -> str:
    """Return current health telemetry (status, latency_ms, error_rate_percent) for a production service.
    Valid services: 'auth-service', 'payment-gateway', 'atlas-worker'."""
    telemetry = {
        "auth-service": {"status": "healthy", "latency_ms": 42, "error_rate_percent": 0.01},
        "payment-gateway": {"status": "degraded", "latency_ms": 1850, "error_rate_percent": 14.5},
        "atlas-worker": {"status": "healthy", "latency_ms": 110, "error_rate_percent": 0.2},
    }
    if service_name not in telemetry:
        return f"Unknown service '{service_name}'. Available: {list(telemetry.keys())}"
    return json.dumps(telemetry[service_name])


@tool
def get_recent_deploys(service_name: str) -> str:
    """Return the recent deployment history for a production service."""
    deploys = {
        "auth-service": [{"version": "v1.9.2", "deployed_ago": "4 days ago", "author": "alice"}],
        "payment-gateway": [
            {"version": "v2.4.0", "deployed_ago": "12 minutes ago", "author": "bob", "notes": "Switch Stripe timeout to 500ms"},
            {"version": "v2.3.9", "deployed_ago": "2 days ago", "author": "carol", "notes": "Routine dependency bump"},
        ],
        "atlas-worker": [{"version": "v0.8.1", "deployed_ago": "1 day ago", "author": "dan"}],
    }
    if service_name not in deploys:
        return f"No deployment history for '{service_name}'."
    return json.dumps(deploys[service_name])


@tool
def get_recent_error_logs(service_name: str) -> str:
    """Fetch the most recent error log entries for a production service."""
    logs = {
        "auth-service": [],
        "payment-gateway": [
            "2026-11-06T16:45:12Z ERROR [req-991] UpstreamTimeoutError: Stripe /v1/charges exceeded 500ms timeout",
            "2026-11-06T16:45:19Z ERROR [req-994] UpstreamTimeoutError: Stripe /v1/charges exceeded 500ms timeout",
        ],
        "atlas-worker": [],
    }
    if service_name not in logs:
        return f"No logs found for '{service_name}'."
    return json.dumps(logs[service_name])

Now we pass the model, the tools list, and a clear `system_prompt` to `create_agent`. Notice how the agent automatically chains multiple tool calls (checking health, deployments, and error logs) before writing its root-cause analysis:

In [ ]:
sre_agent = create_agent(
    model=model,
    tools=[get_service_health, get_recent_deploys, get_recent_error_logs],
    system_prompt=(
        "You are an SRE incident investigator. Always inspect service health, "
        "recent deploys, and error logs before concluding why an incident is happening. "
        "Cite the exact version and error message in your final report."
    ),
)

result = sre_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Checkout is failing for users! Investigate payment-gateway and explain the root cause.",
            }
        ]
    }
)

for msg in result["messages"]:
    msg.pretty_print()

We can even combine tool-calling agents with structured outputs by passing `response_format` to `create_agent`. Once the agent finishes calling tools, the harness populates `result["structured_response"]` with a validated Pydantic object:

In [ ]:
class IncidentDiagnosis(BaseModel):
    """Structured root-cause diagnosis produced after investigating telemetry."""

    service: str = Field(description="Name of the affected service")
    suspect_version: str = Field(description="Deployment version that introduced the issue")
    root_cause: str = Field(description="Technical explanation of the failure based on logs and deploys")
    recommend_rollback: bool = Field(description="True if error rate exceeds 5% after a recent deploy")


structured_sre_agent = create_agent(
    model=model,
    tools=[get_service_health, get_recent_deploys, get_recent_error_logs],
    system_prompt="Investigate the reported service using your tools and produce a structured diagnosis.",
    response_format=IncidentDiagnosis,
)

diag_result = structured_sre_agent.invoke(
    {"messages": [{"role": "user", "content": "Diagnose the current issue on payment-gateway."}]}
)

diagnosis: IncidentDiagnosis = diag_result["structured_response"]
print(diagnosis.model_dump_json(indent=2))

**Exercise 1:** Ask `sre_agent` a broad question such as _"Are any of our services unhealthy right now? Check `auth-service`, `payment-gateway`, and `atlas-worker`."_ Inspect the printed message trace: does the model request all three `get_service_health` calls in parallel in a single turn, or sequentially across turns?

## Playing around with graphs

While `create_agent` is great for open-ended tool loops, production software engineering workflows often need **stronger guarantees**:

- **Deterministic control flow:** always run a validation step or a security check before calling a tool;
- **Custom state:** track typed fields (like retry counts, severity flags, or ticket IDs) alongside the message list;
- **Persistence & memory:** save checkpoints after every step so a conversation can resume across HTTP requests or recover from crashes;
- **Human-in-the-loop (HITL):** pause execution and wait for explicit human approval before executing a destructive action (like rolling back production).

[**LangGraph**](https://docs.langchain.com/oss/python/langgraph/overview) models workflows as **stateful graphs** (state machines).

### State machines

Every LangGraph workflow is built from three core primitives:

1. **State (`TypedDict` or Pydantic model):** a shared data structure representing the current snapshot of the workflow.
2. **Nodes (Python functions):** functions that take the current `state` as input, perform computation (an LLM call, a tool call, or ordinary code), and return a dictionary of **state updates**.
3. **Edges:** transitions between nodes. **Normal edges** (`add_edge`) always go to the same next node; **conditional edges** (`add_conditional_edges`) call a routing function on the current state to decide which node to visit next.

Let's start with a clean, deterministic state machine that classifies an alert based on the Atlas handbook rule (_"If the error rate remains above 5 percent, roll back"_) and routes it accordingly.

In [ ]:
from typing import TypedDict

from IPython.display import Markdown, display
from langgraph.graph import END, START, StateGraph


class AlertState(TypedDict):
    service: str
    error_rate: float
    severity: str
    action_taken: str


def classify_alert(state: AlertState) -> dict:
    """Node 1: classify the alert severity from the error rate."""
    if state["error_rate"] > 5.0:
        return {"severity": "critical"}
    return {"severity": "normal"}


def route_alert(state: AlertState) -> Literal["trigger_rollback", "log_metrics"]:
    """Conditional edge router: decide the next node based on state['severity']."""
    if state["severity"] == "critical":
        return "trigger_rollback"
    return "log_metrics"


def trigger_rollback(state: AlertState) -> dict:
    """Node 2a: handle critical alerts."""
    return {
        "action_taken": f"Initiated rollback for {state['service']} (error rate {state['error_rate']}% > 5%)."
    }


def log_metrics(state: AlertState) -> dict:
    """Node 2b: handle normal alerts."""
    return {
        "action_taken": f"Logged normal operation for {state['service']} (error rate {state['error_rate']}%)."
    }


# Build the state machine
workflow = StateGraph(AlertState)
workflow.add_node("classify_alert", classify_alert)
workflow.add_node("trigger_rollback", trigger_rollback)
workflow.add_node("log_metrics", log_metrics)

workflow.add_edge(START, "classify_alert")
workflow.add_conditional_edges("classify_alert", route_alert)
workflow.add_edge("trigger_rollback", END)
workflow.add_edge("log_metrics", END)

alert_graph = workflow.compile()

print("Critical run:", alert_graph.invoke({"service": "payment-gateway", "error_rate": 14.5}))
print("Normal run:  ", alert_graph.invoke({"service": "auth-service", "error_rate": 0.01}))

Every compiled LangGraph exposes its structure via `get_graph()`, which we can render as a Mermaid diagram:

In [ ]:
from IPython.display import Image

Image(alert_graph.get_graph().draw_mermaid_png())

### Building the tool-calling agent loop explicitly in LangGraph

Now let's demystify `create_agent` by building our own ReAct agent state machine from scratch! We use:

- `MessagesState`: a built-in state schema with a `messages` key configured with an **append reducer** (`add_messages`), so returning `{"messages": [new_msg]}` appends `new_msg` to the conversation history instead of overwriting it;
- an `assistant` node that calls `model_with_tools.invoke(state["messages"])`;
- LangGraph's prebuilt `ToolNode` (which executes any `tool_calls` in the last `AIMessage`) and `tools_condition` (which routes to `"tools"` if `tool_calls` are present, or `END` otherwise).

In [ ]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

sre_tools = [get_service_health, get_recent_deploys, get_recent_error_logs]
llm_with_sre_tools = model.bind_tools(sre_tools)


def assistant_node(state: MessagesState) -> dict:
    """Call the tool-enabled LLM on the current message history."""
    sys_msg = SystemMessage(
        content="You are a concise SRE assistant. Use the provided tools to answer questions accurately."
    )
    response = llm_with_sre_tools.invoke([sys_msg] + state["messages"])
    return {"messages": [response]}


agent_builder = StateGraph(MessagesState)
agent_builder.add_node("assistant", assistant_node)
agent_builder.add_node("tools", ToolNode(sre_tools))

agent_builder.add_edge(START, "assistant")
# If the latest AIMessage has tool_calls -> go to 'tools'; otherwise -> END
agent_builder.add_conditional_edges("assistant", tools_condition)
# After executing tools, always loop back to 'assistant' so the model can observe the results
agent_builder.add_edge("tools", "assistant")

custom_sre_graph = agent_builder.compile()
display(Markdown(f"```mermaid\n{custom_sre_graph.get_graph().draw_mermaid()}\n```"))

### Checkpointing and multi-turn memory (`InMemorySaver`)

In Lab 1, we had to write a custom `ConversationManager` class to store `self.history` between turns. In LangGraph, we simply compile our graph with a **checkpointer** such as [`InMemorySaver`](https://reference.langchain.com/python/langgraph/checkpoints/memory/InMemorySaver) (or `PostgresSaver` / `SqliteSaver` in production).

By passing a `thread_id` in `config={"configurable": {"thread_id": "..."}}`, the graph automatically loads previous messages for that thread and saves every new step:

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
persistent_sre_graph = agent_builder.compile(checkpointer=memory)

thread_config = {"configurable": {"thread_id": "incident-thread-1"}}

# Turn 1: ask about payment-gateway
turn1 = persistent_sre_graph.invoke(
    {"messages": [HumanMessage(content="Check the health of payment-gateway.")]},
    config=thread_config,
)
print("Turn 1 answer:", turn1["messages"][-1].content)

# Turn 2: refer to "that service" — the checkpointer remembers Turn 1!
turn2 = persistent_sre_graph.invoke(
    {"messages": [HumanMessage(content="Who deployed the latest version of that service, and when?")]},
    config=thread_config,
)
print("\nTurn 2 answer:", turn2["messages"][-1].content)

### Human-in-the-loop (`interrupt` and `Command`)

Reading telemetry is safe, but **mutating production** (rolling back a deployment, restarting a database, deleting resources) carries real risk. In Lab 1 we noted that high-impact actions require **human approval**.

LangGraph provides the [`interrupt()`](https://docs.langchain.com/oss/python/langgraph/interrupts) function, which pauses graph execution, persists the exact state in the checkpointer, and surfaces a payload to the caller. Once a human operator reviews the request, we resume the graph from the exact line where it paused using `Command(resume=...)`.

In [ ]:
from langgraph.types import Command, interrupt


class RollbackState(TypedDict):
    service: str
    target_version: str
    approved: bool | None
    status: str


def request_human_approval(state: RollbackState) -> dict:
    """Pause execution and ask a human operator to approve or reject the rollback."""
    decision = interrupt(
        {
            "question": f"Approve production rollback of '{state['service']}' to {state['target_version']}?",
            "service": state["service"],
            "target_version": state["target_version"],
        }
    )
    return {"approved": bool(decision)}


def perform_or_abort_rollback(state: RollbackState) -> dict:
    """Execute the rollback only if the human approved it."""
    if state["approved"]:
        return {"status": f"SUCCESS: Rolled back {state['service']} to {state['target_version']}."}
    return {"status": f"ABORTED: Rollback of {state['service']} was rejected by the operator."}


rb_builder = StateGraph(RollbackState)
rb_builder.add_node("request_human_approval", request_human_approval)
rb_builder.add_node("perform_or_abort_rollback", perform_or_abort_rollback)
rb_builder.add_edge(START, "request_human_approval")
rb_builder.add_edge("request_human_approval", "perform_or_abort_rollback")
rb_builder.add_edge("perform_or_abort_rollback", END)

# A checkpointer is required for interrupts so LangGraph can save state while paused!
rollback_graph = rb_builder.compile(checkpointer=InMemorySaver())

rb_config = {"configurable": {"thread_id": "rollback-payment-42"}}

# Step 1: Start the workflow — it will pause at `interrupt(...)`
paused_state = rollback_graph.invoke(
    {"service": "payment-gateway", "target_version": "v2.3.9", "approved": None, "status": "pending"},
    config=rb_config,
)
print("Graph paused! Interrupt payload:")
print(paused_state["__interrupt__"])

# Step 2: Simulate the on-call engineer approving the rollback (resume=True)
resumed_state = rollback_graph.invoke(Command(resume=True), config=rb_config)
print("\nResumed final state:")
print(resumed_state)

### Mission 2 &mdash; Build a retry-bounded verification loop in LangGraph (guided)

**Goal:** prevent an automated code-fix or config-generation loop from running forever when validation fails.

One of the biggest risks with autonomous loops is an **unbounded retry loop** (burning tokens endlessly when a test keeps failing). Complete the two blanks (`...`) in `route_after_validation` below so that:

- if `state["is_valid"]` is `True`, the graph transitions to `END`;
- if `state["attempts"] >= state["max_attempts"]`, the graph also stops (`END`) instead of looping forever;
- otherwise, it routes back to `"generate_candidate"`.

Then run `check_mission_2()` to verify both the passing and max-retry-exceeded paths.

In [ ]:
class ConfigGenState(TypedDict):
    target_port: int
    attempts: int
    max_attempts: int
    candidate_config: str
    is_valid: bool


def generate_candidate(state: ConfigGenState) -> dict:
    attempt = state["attempts"] + 1
    # Simulate a generator that only gets the port right on attempt 2
    port = state["target_port"] if attempt >= 2 else -1
    return {"attempts": attempt, "candidate_config": f"listen_port={port}"}


def validate_candidate(state: ConfigGenState) -> dict:
    valid = f"listen_port={state['target_port']}" in state["candidate_config"] and state["target_port"] > 0
    return {"is_valid": valid}


def route_after_validation(state: ConfigGenState) -> str:
    # TODO: return END if the config is valid OR if attempts >= max_attempts; otherwise "generate_candidate"
    if ...:
        return END
    return ...


def check_mission_2():
    g = StateGraph(ConfigGenState)
    g.add_node("generate_candidate", generate_candidate)
    g.add_node("validate_candidate", validate_candidate)
    g.add_edge(START, "generate_candidate")
    g.add_edge("generate_candidate", "validate_candidate")
    g.add_conditional_edges("validate_candidate", route_after_validation)
    app = g.compile()

    # Case 1: succeeds on attempt 2
    ok = app.invoke({"target_port": 8080, "attempts": 0, "max_attempts": 3, "candidate_config": "", "is_valid": False})
    assert ok["is_valid"] is True and ok["attempts"] == 2, f"Unexpected state for ok run: {ok}"

    # Case 2: impossible port (-99) must stop at max_attempts=3 instead of looping forever
    fail = app.invoke({"target_port": -99, "attempts": 0, "max_attempts": 3, "candidate_config": "", "is_valid": False})
    assert fail["is_valid"] is False and fail["attempts"] == 3, f"Loop did not stop at max_attempts: {fail}"
    print("Mission 2 check passed! Both success and bounded-failure paths work.")


# Complete the TODO in route_after_validation, then uncomment the next line:
# check_mission_2()

## Deep agents

In Lab 2, we built a classic **RAG pipeline**: chunk documents ahead of time, compute TF-IDF / vector similarity, and inject the top-$k$ chunks into a single prompt. That works well for simple factual lookups, but what if a software engineering task requires:

- browsing a directory tree (`ls`, `glob`) to discover what files exist;
- searching across files with regex (`grep`) and reading full files (`read_file`);
- delegating focused investigations to isolated **subagents** (`task`) so intermediate file reads don't clutter the main context window; and
- writing a synthesized report (`write_file`) into a workspace?

This architecture &mdash; combining an LLM with built-in filesystem tools, context-compaction middleware, and subagent delegation &mdash; is what LangChain calls a [**Deep Agent**](https://docs.langchain.com/oss/python/deepagents/overview) (via the `deepagents` library).

### Creating a research agent

Let's build a **Deep Agent** that researches the **Atlas project handbook** from Lab 2 (`data/rag/architecture.md`, `operations.md`, `security.md`).

Instead of giving the agent unrestricted access to our host machine, we follow the principle of **least privilege** using `deepagents.backends` and `FilesystemPermission`:

1. **`FilesystemBackend(root_dir=..., virtual_mode=True)`**: mounts `./data/rag` under a virtual root (`virtual_mode=True` blocks `..` path traversal outside `root_dir`).
2. **`CompositeBackend`**: routes `/handbook/` to our local `./data/rag` directory, while routing everything else (like `/reports/`) to an ephemeral in-memory `StateBackend()`.
3. **`FilesystemPermission`**: explicitly denies `write` operations on `/handbook/**` so the agent cannot modify or corrupt the source documentation.

In [ ]:
from pathlib import Path

from deepagents import FilesystemPermission, SubAgent, create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend

HANDBOOK_DIR = Path.cwd() / "data" / "rag"

backend = CompositeBackend(
    # Default route: ephemeral in-memory filesystem (safe for scratch notes and reports)
    default=StateBackend(),
    # Mount the local Atlas handbook directory at /handbook/
    routes={
        "/handbook/": FilesystemBackend(root_dir=HANDBOOK_DIR, virtual_mode=True),
    },
)

# Verify what the backend sees under /handbook/
for entry in backend.ls("/handbook/").entries:
    print(entry["path"], f"({entry['size']} bytes)")

Now we create the research agent with `create_deep_agent`. By default, `create_deep_agent` equips the model with:

- `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` (backed by our `CompositeBackend`);
- `task` (for spawning subagents with isolated context windows);
- automatic context summarization middleware for long trajectories.

Let's ask our research agent to explore `/handbook/`, cross-reference all three documents (`architecture.md`, `operations.md`, `security.md`), save a compliance checklist to `/reports/go_live_checklist.md`, and summarize its findings:

In [ ]:
research_agent = create_deep_agent(
    model=model,
    backend=backend,
    permissions=[
        # Allow reading anywhere, but forbid modifying the handbook source files!
        FilesystemPermission(operations=["read"], paths=["/**"], mode="allow"),
        FilesystemPermission(operations=["write"], paths=["/handbook/**"], mode="deny"),
        FilesystemPermission(operations=["write"], paths=["/reports/**"], mode="allow"),
    ],
    system_prompt=(
        "You are a technical research agent auditing the Atlas project. "
        "The official documentation is mounted read-only at `/handbook/`. "
        "Use `ls`, `grep`, or `read_file` to inspect the handbook files. "
        "Never invent policies not stated in `/handbook/`. Always cite the exact file name."
    ),
)

research_result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Inspect the documentation in `/handbook/`. Write a concise Markdown checklist "
                    "to `/reports/go_live_checklist.md` covering: (1) production database & queue requirements, "
                    "(2) deployment & rollback rules, and (3) authentication & secret management rules. "
                    "Then reply with a short summary."
                ),
            }
        ]
    }
)

print("=== Final Agent Reply ===")
print(research_result["messages"][-1].content)

Because `/reports/` is backed by `StateBackend`, the file written by the agent is stored directly in `research_result["files"]` without polluting our local git repository! Let's inspect the generated file:

In [ ]:
files_in_state = research_result.get("files", {})
print("Files written to StateBackend:", list(files_in_state.keys()))

if "/reports/go_live_checklist.md" in files_in_state:
    checklist_content = "\n".join(files_in_state["/reports/go_live_checklist.md"]["content"])
    display(Markdown(checklist_content))

### Mission 3 &mdash; Configure a specialized subagent and permission rule (guided)

**Goal:** delegate focused security audits to a dedicated `SubAgent` and enforce read-only permissions.

When a deep agent tackles a large repository, reading dozens of files in the main agent's loop quickly fills up its context window. By declaring `subagents=[SubAgent(...)]`, the main agent can invoke the `task` tool to spawn a subagent with its **own isolated context window**, which reads the necessary files and returns only a concise summary to the parent agent.

Complete the blanks (`...`) below to:

1. Define a `SubAgent` named `"security-auditor"` with a description and system prompt focused on `security.md` and dependency advisories;
2. Add a `FilesystemPermission` rule that denies `"write"` operations across `"/**"` so the subagent is strictly read-only;
3. Run `check_mission_3()` to validate your configuration.

In [ ]:
# TODO 1: create a read-only FilesystemPermission that denies "write" on "/**"
deny_all_writes = FilesystemPermission(
    operations=[...],
    paths=["/**"],
    mode=...,
)

# TODO 2: define the declarative SubAgent specification
security_subagent = SubAgent(
    name="...",
    description="Audits Atlas security policies, authentication, secrets, and dependency vulnerabilities.",
    system_prompt=(
        "You are the Atlas security auditor. Read `/handbook/security.md` and use "
        "`get_package_advisory` to check whether dependencies block a release."
    ),
    tools=[get_package_advisory],
    permissions=[deny_all_writes],
)


def check_mission_3():
    assert deny_all_writes.operations == ["write"], "operations should be ['write']"
    assert deny_all_writes.mode == "deny", "mode should be 'deny'"
    assert security_subagent["name"] == "security-auditor", "SubAgent name must be 'security-auditor'"
    assert "permissions" in security_subagent and deny_all_writes in security_subagent["permissions"]

    # Verify the deep agent compiles cleanly with the subagent attached
    lead_agent = create_deep_agent(
        model=model,
        backend=backend,
        subagents=[security_subagent],
        system_prompt="Delegate security and dependency questions to the `security-auditor` subagent.",
    )
    assert lead_agent is not None
    print("Mission 3 check passed! Lead deep agent compiled with `security-auditor` subagent.")


# Complete the TODOs above, then uncomment the next line:
# check_mission_3()

**Exercise 2 (RAG vs. Agentic Search):** Compare the vector-retrieval approach from [Lab 2](lab2_rag.ipynb) with the Deep Agent's filesystem tools (`ls`, `grep`, `read_file`) in Lab 3:

1. When is pre-indexed vector RAG better than letting an agent run `grep` and `read_file`? (Consider corpus size, latency, and token cost.)
2. When is agentic file navigation (`ls` / `grep` / `read_file`) better than top-$k$ chunk retrieval? (Consider cross-file reasoning, exact symbol lookups in source code, and chunk boundary issues.)
3. How could you combine both in a single agent for your final TSIS project?

## Takeaways

- **Provider-agnostic interfaces (`BaseChatModel`, `init_chat_model`)** decouple your application logic from any single LLM vendor.
- **`@tool` and `with_structured_output`** turn Python functions and Pydantic models into machine-validated contracts with minimal boilerplate.
- **Agents (`create_agent`)** run an LLM in a tool-calling loop (_Reason $\rightarrow$ Act $\rightarrow$ Observe_), which is powerful for exploration but requires guardrails.
- **State machines (`LangGraph`)** let you combine deterministic code with LLM nodes, enforce retry bounds, persist multi-turn memory (`InMemorySaver`), and pause for human approval (`interrupt` / `Command`) before high-risk actions.
- **Deep Agents (`create_deep_agent`)** pair an agent loop with virtual filesystems (`CompositeBackend`), explicit access rules (`FilesystemPermission`), and isolated `SubAgent`s to tackle multi-file research and software engineering tasks safely.